intro...

---
# 1. Setup

In [2]:
# install dependencies
!pip install -qq transformer_lens einops matplotlib

In [4]:
%%capture
import torch
import numpy as np
import einops
import matplotlib.pyplot as plt
import os
import warnings
from transformer_lens import HookedTransformer

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)

In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cpu


In [ ]:
# load GPT-2 Small: ~163M params, 12 layers, 12 heads, d_model=768
# standard sandbox model for mechanistic interpretability research.
model = HookedTransformer.from_pretrained(
    'gpt2',
    center_unembed = True,
    center_writing_weights = True,
    fold_ln = True,
    refactor_factored_attn_matrices = True
).to(DEVICE)
model.eval()

n_layers = model.cfg.n_layers
n_heads  = model.cfg.n_heads
d_model  = model.cfg.d_model

print(f'Layers: {n_layers}, Heads: {n_heads}, d_model: {d_model}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cpu
Layers: 12, Heads: 12, d_model: 768
Total parameters: 163,049,041


In [14]:
# sanity check
prompt = 'The Eiffel Tower is located in the city of'
logits, cache = model.run_with_cache(prompt)
probs = torch.softmax(logits[0, -1], dim = -1)
top5 = torch.topk(probs, 5)
print('Top 5 predictions:')
for tok, p in zip(top5.indices, top5.values):
    token_str = model.to_single_str_token(tok.item())
    print(f'  {repr(token_str):15s}  {p.item():.4f}')

Top 5 predictions:
  ' London'        0.0691
  ' Paris'         0.0688
  ' Amsterdam'     0.0403
  ' Berlin'        0.0323
  ' New'           0.0279


> Note: GPT-2 Small ranks London marginally above Paris here. This is due to model's limited factual associations at this very small scale, not an error.

---
# 2. Attention Pattern Visualization